In [1]:
# ==============================================================================
# CONCEPT EXPLORATION: WHY UNSLOTH?
# Standard fine-tuning uses Hugging Face's native PyTorch loops, which keep giant
# intermediate computational tensors in VRAM (activation memory). 
# Unsloth rewrites the core mathematical kernels (triton kernels) of the attention 
# mechanisms. This reduces memory overhead by up to 60% and accelerates training 
# by 2x to 5x without losing any model accuracy.
# ==============================================================================

import sys
import torch

# Check GPU capability to determine whether to install the Ampere (A100/H100) or Turing/Volta (T4/V100) architecture build.
# Kaggle T4 provides Compute Capability 7.5, which requires the standard Unsloth wheel.
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    # Ampere and newer GPUs (e.g., A100, RTX 3090/4090)
    !pip install -q --no-deps xformers trl peft lora_ctl_network
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # Pre-Ampere GPUs (e.g., NVIDIA T4 on Kaggle)
    !pip install -q --no-deps xformers trl peft lora_ctl_network
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install fundamental data processing libraries
!pip install -q pandas scikit-learn datasets

print(">>> Environment successfully initialized for Unsloth training pipeline.")

ERROR: Could not find a version that satisfies the requirement lora_ctl_network (from versions: none)
ERROR: No matching distribution found for lora_ctl_network
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.8 MB/

In [2]:
# ==============================================================================
# CONCEPT EXPLORATION: QUANTIZATION & WEIGHTS
# - Weights: Deep learning models are giant matrices of numbers (parameters or weights)
#   that determine how input signals map to output signals.
#   Normally, these are stored in 16-bit floating point precision (FP16 or BF16),
#   requiring 2 bytes of memory per parameter. A 1.5B model requires ~3GB of VRAM just to load.
# - Quantization: This process downscales the numerical precision of weights (e.g., 
#   from 16-bit to a highly compact 4-bit configuration called NormalFloat4 or NF4). 
#   This compresses our 1.5B parameter model's memory footprint to ~1.2GB, ensuring 
#   ample headroom remains inside the Kaggle GPU container.
# ==============================================================================

from unsloth import FastLanguageModel
import torch

# Configuration Variables
MAX_SEQUENCE_LENGTH = 1024 # Restricting total context window length to conserve local memory footprints
DATA_TYPE = None           # None automatically detects system hardware capabilities (Float16 for T4, Bfloat16 for Ampere)
LOAD_IN_4BIT = True        # True activates 4-bit Quantization to optimize memory overhead

# Fetching the optimized Qwen2.5-Coder base model via Unsloth's registry
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct", 
    max_seq_length = MAX_SEQUENCE_LENGTH,
    dtype = DATA_TYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

print(f">>> Base Model loaded into VRAM. Current allocation: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

>>> Base Model loaded into VRAM. Current allocation: 1.19 GB


In [3]:
# ==============================================================================
# CONCEPT EXPLORATION: LoRA vs. QLoRA
# - Full Fine-Tuning: Modifies every parameter in the model. This requires vast 
#   amounts of VRAM because optimizer algorithms must store gradient states for all weights.
# - LoRA (Low-Rank Adaptation): Freezes the original heavy weight matrices ($W$). Instead, 
#   it injects small, trainable pairs of rank-decomposition matrices ($A$ and $B$) right 
#   next to them. Only these thin layers change during training, drastically cutting 
#   down the parameters you need to update.
# - QLoRA (Quantized LoRA): Combines LoRA with a 4-bit quantized base model. The base 
#   model is completely locked in ultra-compact 4-bit precision, while the trainable 
#   LoRA matrices operate in standard 16-bit precision to maintain maximum accuracy.
#
# - Target Modules: We target ALL linear projections (q, k, v, o, gate, up, down). 
#   This prevents "Catastrophic Forgetting" across your three dataset groups by distributing 
#   the learning adjustments evenly throughout the entire transformer structure.
# ==============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Rank: Controls the structural width of the adaptation matrices. 16 provides an ideal balance of capacity and control.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],                     # Targeting all linear projection layers to ensure comprehensive system coverage.
    lora_alpha = 32,       # Alpha: A scaling factor that regulates the influence of the LoRA weights relative to the base model.
    lora_dropout = 0,      # Optimized to 0 for Unsloth training paths to maximize performance speed.
    bias = "none",         # Retaining a "none" configuration avoids tracking unnecessary gradient offsets.
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM by discarding intermediate layer states and recomputing them during backpropagation.
    random_state = 3407,   # Static seeding guarantees reproducible training runs.
    use_rslora = False,    # Rank-Stabilized LoRA is omitted here to favor standard, predictable convergence behaviors.
    loftq_config = None,   # LoftQ initialization is skipped since we are using pre-quantized base configurations.
)

print(">>> QLoRA adapters successfully injected into targeted linear projection paths.")

Unsloth 2026.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


>>> QLoRA adapters successfully injected into targeted linear projection paths.


In [4]:
# ==============================================================================
# CONCEPT EXPLORATION: CHATML & TEMPORAL CONTEXT
# Raw text datasets mean very little to standard language models unless they are 
# properly structured into a predictable prompt format. ChatML uses explicit tokens 
# (<|im_start|> and <|im_end|>) to define roles clearly: System, User, and Assistant.
# By baking the 'current_date' and 'current_time' values right into the System prompt, 
# the model establishes a clear baseline for processing relative time expressions 
# (like "at noon" or "tomorrow") and mapping them into the required JSON properties.
# ==============================================================================

import pandas as pd
from datasets import Dataset

TEST_MODE = False  # Set to False when you are ready to run the full 2,750 rows

# Define the exact System Framework Prompt that guides the model's behavior
SYSTEM_TEMPLATE = """
You are a precise macOS system routing core. Your only objective is to translate natural language commands into a single, minimized, valid JSON object.

Strict Rules:
1. Output ONLY raw, valid JSON. Do not include markdown code blocks (```json), conversational text, or explanations.
2. Select the specific "app" and "action" that matches the user's explicit intent.
3. If the command is ambiguous, asks for conversational fluff, or requests an action outside your capabilities, you MUST route to the fallback trigger exactly: {{"app":"router","action":"do-nothing","params":{{}}}}

Output Shape Example:
{{"app":"calendar","action":"open","params":{{}}}}

System Context:
- Current Date: {current_date}
- Current Time: {current_time}
"""

def transform_csv_to_chatml(row):
    """
    Transforms raw columns into a structured conversation layout that matches
    the pre-training design of Qwen2.5-Coder.
    """
    formatted_prompt = (
        f"<|im_start|>system\n{SYSTEM_TEMPLATE.format(current_date=row['current_date'], current_time=row['current_time'])}<|im_end|>\n"
        f"<|im_start|>user\n{row['command']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['json']}<|im_end|>"
    )
    return {"text": formatted_prompt}

# 1. Simulating the loading path from your Kaggle environment directory
# Replace 'dataset.csv' with the actual path to your uploaded dataset file
try:
    df = pd.read_csv("/kaggle/input/datasets/puruthakkar/command-to-agent-action-3-1/dataset.csv")
except FileNotFoundError:
    # Mocking sample structure for execution reference if dataset file isn't present yet
    print(">>> Warning: 'dataset.csv' not found. Using inline fallback examples for verification.")
    mock_data = [
        {"command": "Could you launch Finder?", "json": '{"app":"finder","action":"open","params":{}}', "current_date": "2019-05-10", "current_time": "20:42:36"},
        {"command": "Add a lunch appointment with Sarah at noon", "json": '{"app":"calendar","action":"create_event","params":{"title":"Lunch with Sarah","date":"2024-03-25","time":"12:00:00","duration_in_minutes":60}}', "current_date": "2006-04-17", "current_time": "23:20:19"},
        {"command": "open instagram", "json": '{"app":"router","action":"do-nothing","params":{}}', "current_date": "2010-06-03", "current_time": "22:12:23"}
    ]
    df = pd.DataFrame(mock_data)

# 1. Simulating the loading path from your Kaggle environment directory
# Replace 'dataset.csv' with the actual path to your uploaded dataset file
try:
    eval_df = pd.read_csv("/kaggle/input/datasets/puruthakkar/command-to-agent-action-eval-2/test_dataset.csv")
except FileNotFoundError:
    # Mocking sample structure for execution reference if dataset file isn't present yet
    print(">>> Warning: 'test_dataset.csv' not found. Using inline fallback examples for verification.")
    mock_eval_data = [
        {"command": "Could you launch Finder?", "json": '{"app":"finder","action":"open","params":{}}', "current_date": "2019-05-10", "current_time": "20:42:36"},
        {"command": "Add a lunch appointment with Sarah at noon", "json": '{"app":"calendar","action":"create_event","params":{"title":"Lunch with Sarah","date":"2024-03-25","time":"12:00:00","duration_in_minutes":60}}', "current_date": "2006-04-17", "current_time": "23:20:19"},
        {"command": "open instagram", "json": '{"app":"router","action":"do-nothing","params":{}}', "current_date": "2010-06-03", "current_time": "22:12:23"}
    ]
    eval_df = pd.DataFrame(mock_eval_data)

# test mode config
if TEST_MODE:
    print(f"\n>>> [TEST MODE ACTIVE] Truncating dataset from {len(df)} rows to the first 50 rows.")
    df = df.head(50)

# 2. Map the structural conversions over the dataframe row-by-row
processed_rows = df.apply(transform_csv_to_chatml, axis=1).tolist()
hf_dataset = Dataset.from_list(processed_rows)

eval_processed_rows = eval_df.apply(transform_csv_to_chatml, axis=1).tolist()
hf_eval_dataset = Dataset.from_list(eval_processed_rows)


# Confirming processing integrity by printing example row index 0
print("\n>>> Sample Formatted Training Record Structure:")
print(hf_dataset[0:2]["text"])

print("\n>>> Sample Formatted Eval Record Structure:")
print(hf_eval_dataset[0:2]["text"])


>>> Sample Formatted Training Record Structure:
['<|im_start|>system\n\nYou are a precise macOS system routing core. Your only objective is to translate natural language commands into a single, minimized, valid JSON object.\n\nStrict Rules:\n1. Output ONLY raw, valid JSON. Do not include markdown code blocks (```json), conversational text, or explanations.\n2. Select the specific "app" and "action" that matches the user\'s explicit intent.\n3. If the command is ambiguous, asks for conversational fluff, or requests an action outside your capabilities, you MUST route to the fallback trigger exactly: {"app":"router","action":"do-nothing","params":{}}\n\nOutput Shape Example:\n{"app":"calendar","action":"open","params":{}}\n\nSystem Context:\n- Current Date: Wednesday, 1991-08-14\n- Current Time: 00:36:54\n<|im_end|>\n<|im_start|>user\nI would appreciate it if you could open the fin der tomorrow morning.<|im_end|>\n<|im_start|>assistant\n{"app":"finder","action":"open","params":{}}<|im_en

In [5]:
import torch
from trl import SFTTrainer, SFTConfig 


# Initialize the trainer module
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,          # Using the training split
    eval_dataset = hf_eval_dataset,            # Added the eval split
    
    args = SFTConfig(  
        dataset_text_field = "text",       
        max_length = MAX_SEQUENCE_LENGTH,  
        dataset_num_proc = 2,              
        packing = False,                   
        
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,   
        warmup_steps = 10,                 
        num_train_epochs = 2,              
        learning_rate = 2e-4,              
        fp16 = not torch.cuda.is_bf16_supported(),  
        bf16 = torch.cuda.is_bf16_supported(),      
        
        # --- CONCISE PERFORMANCE LOGGING CONFIGURATION ---
        logging_strategy = "steps",
        logging_steps = 10,                 # Bumped from 5 to 10 to reduce Kaggle output clutter
        eval_strategy = "steps",            # Evaluates periodically during the run
        eval_steps = 50,                    # Calculates validation loss every ~50 steps (~10 times total)
        per_device_eval_batch_size = 4,     # Micro-batch size for validation pass
        # ----------------=================================

        optim = "adamw_8bit",              
        weight_decay = 0.01,               
        lr_scheduler_type = "cosine",      
        seed = 3407,
        output_dir = "training_checkpoints",
    ),
)

# Start the training process
print(">>> Initializing fine-tuning loop...")
trainer_stats = trainer.train()
print(">>> Training successfully completed.")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2373 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/74 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
>>> Initializing fine-tuning loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,373 | Num Epochs = 2 | Total steps = 298
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.294619,0.233342
100,0.272549,0.221306
150,0.248930,0.224250
200,0.234984,0.225082
250,0.227590,0.225366
298,0.226276,0.225001


Unsloth: Restored added_tokens_decoder metadata in training_checkpoints/checkpoint-298/tokenizer_config.json.


>>> Training successfully completed.


In [6]:
mock_data = [
    {"command": "Could you launch Finder?", "json": '{"app":"finder","action":"open","params":{}}', "current_date": "2019-05-10", "current_time": "20:42:36"},
    {"command": "write a note titles writings", "json": '{"app":"calendar","action":"create_event","params":{"title":"Lunch with Sarah","date":"2024-03-25","time":"12:00:00","duration_in_minutes":60}}', "current_date": "2020-03-25", "current_time": "09:20:19"},
    {"command": "start arijit singh on spotify", "json": '{"app":"router","action":"do-nothing","params":{}}', "current_date": "2010-06-03", "current_time": "22:12:23"},
    {"command": "remind me to flower the plants tomorrow 10 am", "json": '{"app":"router","action":"do-nothing","params":{}}', "current_date": "2010-06-03", "current_time": "22:12:23"},
    {"command": "Create a note titled Project kickoff summary with the content Meeting date 2004-03-31 at 11:36:04, discuss goals, assign next steps, and send follow up reminders.", "json": '{"app":"router","action":"do-nothing","params":{}}', "current_date": "2004-06-03", "current_time": "22:12:23"}
]

# 2. Define the Inference Prompt Generator 
# Crucial: This stops exactly where the model needs to start generating
def transform_csv_to_chatml_inference(row):
    formatted_prompt = (
        f"<|im_start|>system\n{SYSTEM_TEMPLATE.format(current_date=row['current_date'], current_time=row['current_time'])}<|im_end|>\n"
        f"<|im_start|>user\n{row['command']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    return formatted_prompt

# 3. Setup Model for Inference
# Based on your previous error, you are likely using Unsloth. 
# Unsloth requires calling this to enable 2x faster inference. If not using Unsloth, replace with model.eval()
if "FastLanguageModel" in globals() or "FastLanguageModel" in locals():
    FastLanguageModel.for_inference(model)
else:
    model.eval()

# 4. Run the Test Loop
print(">>> Starting Model Inference Testing... \n" + "="*50)

for idx, row in enumerate(mock_data):
    # Construct the input prompt
    prompt = transform_csv_to_chatml_inference(row)
    
    # Tokenize input and move to GPU
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Generate tokens
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,        # Adjust based on expected max length of your JSON output
            use_cache=True,            # Speeds up generation
            temperature=0.1,           # Low temperature forces structure/determinism for JSON tasks
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode the outputs. We skip special tokens or keep them to verify ChatML closure tags.
    # We slice outputs to remove the prompt tokens and only show what the model generated.
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    prediction = tokenizer.decode(generated_tokens, skip_special_tokens=False)
    
    # Output evaluation results
    print(f"Test Case #{idx + 1}")
    print(f"Command Sent: {row['command']}")
    print("-" * 50)
    print(f"Expected Target JSON:\n{row['json']}")
    print("-" * 50)
    print(f"Model Predicted Output:\n{prediction}")
    print("=" * 50)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


>>> Starting Model Inference Testing... 


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case #1
Command Sent: Could you launch Finder?
--------------------------------------------------
Expected Target JSON:
{"app":"finder","action":"open","params":{}}
--------------------------------------------------
Model Predicted Output:
{"app":"finder","action":"open","params":{}}<|im_end|>


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case #2
Command Sent: write a note titles writings
--------------------------------------------------
Expected Target JSON:
{"app":"calendar","action":"create_event","params":{"title":"Lunch with Sarah","date":"2024-03-25","time":"12:00:00","duration_in_minutes":60}}
--------------------------------------------------
Model Predicted Output:
{"app":"notes","action":"create_note","params":{"title":"writings"}}<|im_end|>


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case #3
Command Sent: start arijit singh on spotify
--------------------------------------------------
Expected Target JSON:
{"app":"router","action":"do-nothing","params":{}}
--------------------------------------------------
Model Predicted Output:
{"app":"spotify","action":"search_and_play","params":{"query":"arijit singh","scope":"artist"}}<|im_end|>


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Case #4
Command Sent: remind me to flower the plants tomorrow 10 am
--------------------------------------------------
Expected Target JSON:
{"app":"router","action":"do-nothing","params":{}}
--------------------------------------------------
Model Predicted Output:
{"app":"reminders","action":"create_reminder","params":{"title":"flower the plants","date":"2010-06-04","time":"10:00:00"}}<|im_end|>
Test Case #5
Command Sent: Create a note titled Project kickoff summary with the content Meeting date 2004-03-31 at 11:36:04, discuss goals, assign next steps, and send follow up reminders.
--------------------------------------------------
Expected Target JSON:
{"app":"router","action":"do-nothing","params":{}}
--------------------------------------------------
Model Predicted Output:
{"app":"notes","action":"create_note","params":{"title":"Project kickoff summary","content":"Meeting date 2004-03-31 at 11:36:04, discuss goals, assign next steps, and send follow up reminders."}}<|im_end|

In [7]:
# ==============================================================================
# CONCEPT EXPLORATION: ADAPTER EXPORT vs. FULL MERGE
# Instead of modifying and saving the entire multi-gigabyte model, this step exports 
# only the highly optimized LoRA adapters (the tiny weight matrices we trained). 
# This produces a very lightweight asset (~50MB to ~100MB) that contains all of your 
# specific function-routing behaviors. You can upload this directly to Hugging Face, 
# making it easy to download and merge into the base model on your local client machine later.
# ==============================================================================

import os
import shutil

OUTPUT_DIRECTORY = "qwen_macos_agent_lora_3-1"

# Save the trained LoRA adapters and tokenizer configurations to the workspace directory
model.save_pretrained(OUTPUT_DIRECTORY)
tokenizer.save_pretrained(OUTPUT_DIRECTORY)

print(f">>> Optimization adapters saved successfully to directory: '{OUTPUT_DIRECTORY}'")
print(">>> This folder contains 'adapter_config.json' and 'adapter_model.safetensors'.")

# Compress the output directory into a separate zip file
# shutil.make_archive automatically appends the '.zip' extension to the base_name
shutil.make_archive(base_name=OUTPUT_DIRECTORY, format='zip', root_dir=OUTPUT_DIRECTORY)

print(f">>> Directory successfully zipped to: '{OUTPUT_DIRECTORY}.zip'")
print(">>> You can now upload this directory or the zip archive directly to Hugging Face whenever you are ready.")

Unsloth: Restored added_tokens_decoder metadata in qwen_macos_agent_lora_3-1/tokenizer_config.json.


>>> Optimization adapters saved successfully to directory: 'qwen_macos_agent_lora_3-1'
>>> This folder contains 'adapter_config.json' and 'adapter_model.safetensors'.
>>> Directory successfully zipped to: 'qwen_macos_agent_lora_3-1.zip'
>>> You can now upload this directory or the zip archive directly to Hugging Face whenever you are ready.
